# Qwen 3.5 + MCP Tool Calling

A genuine **Model Context Protocol (MCP)** host that connects Qwen 3.5 to MCP tool servers.

```
┌──────────────┐    MCP (stdio)     ┌────────────────────┐
│  MCP Host    │ ◄────────────────► │  MCP Weather Server │
│  (mcp_host)  │                    │  (tools/*.py)        │
│              │    Ollama REST     └────────────────────┘
│  Qwen 3.5    │ ◄──────────────►  localhost:11434
└──────────────┘
```

**Flow:** User query → LLM (with tool schemas) → LLM emits tool_call → MCP executes tool → result back to LLM → final answer

In [1]:
## Cell 1: Config
MODEL_NAME = "qwen3.5:35b"   # <<< Change model here
MCP_SERVER_SCRIPT = "tools/mcp_weather_server.py"

from mcp_host import MCPHost
print(f"Model : {MODEL_NAME}")
print(f"Server: {MCP_SERVER_SCRIPT}")

Model : qwen3.5:35b
Server: tools/mcp_weather_server.py


In [2]:
## Cell 2: Start host & connect to MCP server
host = MCPHost(model=MODEL_NAME, verbose=True)
host.start()
host.connect_server("weather", "python3", [MCP_SERVER_SCRIPT])

print(f"\nDiscovered tools: {host.tool_names}")
print(f"\nOllama-format schemas:")
for schema in host.tool_schemas:
    f = schema["function"]
    print(f"  • {f['name']}: {f['description'][:80]}")

✅ MCPHost started (background event-loop running)
  Registered tool: get_current_weather (from 'weather')
  Registered tool: get_weather_forecast (from 'weather')
✅ Connected to MCP server 'weather' — 2 tool(s)

Discovered tools: ['get_current_weather', 'get_weather_forecast']

Ollama-format schemas:
  • get_current_weather: Get the current weather for a given location using the Open-Meteo API.

    Args
  • get_weather_forecast: Get a weather forecast for the next N days using the Open-Meteo API.

    Args:



## Test 1: Weather query → should trigger `get_current_weather` tool call

In [3]:
## Cell 3: Weather query — LLM calls get_current_weather via MCP
answer = host.chat("What's the weather like in San Diego right now?")


👤 User: What's the weather like in San Diego right now?

💭 Thinking: The user is asking about the current weather in San Diego. I should use the get_current_weather function to get this information. The required parameter is "location" which should be "San Diego". I don't need to specify the unit since it has a default value of fahrenheit....

🔧 Tool call [1]: get_current_weather({"location": "San Diego"})
📦 Result: {"location": "San Diego, United States", "coordinates": {"lat": 32.71571, "lon": -117.16472}, "temperature": 79.8, "feels_like": 79.2, "unit": "fahrenheit", "condition": "Overcast", "humidity": 39, "wind_speed": 8.1, "wind_direction": 246, "timezone": "America/Los_Angeles"}

💭 Thinking: Great! I got the current weather information for San Diego. Let me present this information in a clear and helpful way for the user....

🤖 Answer: The current weather in San Diego is:

- **Temperature**: 79.8°F (feels like 79.2°F)
- **Condition**: Overcast
- **Humidity**: 39%
- **Wind**: 8.

## Test 2: Forecast query → should trigger `get_weather_forecast`

In [ ]:
## Cell 4: Forecast query — LLM should pick get_weather_forecast
answer = host.chat("Give me a 5-day weather forecast for Tokyo in celsius.")


👤 User: Give me a 5-day weather forecast for Tokyo in celsius.

💭 Thinking: The user is asking for a 5-day weather forecast for Tokyo in Celsius. I need to use the get_weather_forecast function with the following parameters:
- location: "Tokyo"
- days: 5
- unit: "celsius"

Let me make this function call....

🔧 Tool call [1]: get_weather_forecast({"location": "Tokyo", "days": 5, "unit": "celsius"})
📦 Result: {"location": "Tokyo, Japan", "forecast": [{"date": "2026-03-01", "temp_high": 15.7, "temp_low": 6.3, "unit": "celsius", "condition": "Partly cloudy", "precipitation_chance": 0, "wind_speed_max": 8.4}, {"date": "2026-03-02", "temp_high": 14.1, "temp_low": 4.6, "unit": "celsius", "condition": "Partly cloudy", "precipitation_chance": 26, "wind_speed_max": 5.2}, {"date": "2026-03-03", "temp_high": 8.2, "temp_low": 0.5, "unit": "celsius", "condition": "Moderate rain", "precipitation_chance": 75, "wind_speed_max": 9.2}, {"date": "2026-03-04", "temp_high": 15.7, "temp_low": 0.8, "unit": "

## Test 3: Non-tool query → LLM should answer directly (no tool call)

In [ ]:
## Cell 5: Non-tool query — should NOT trigger any tool call
answer = host.chat("What is the capital of France?")


👤 User: What is the capital of France?

💭 Thinking: The user is asking a general knowledge question about the capital of France. This doesn't require any weather-related tools or API calls. I can answer this directly from my knowledge.

The capital of France is Paris. This is a well-established fact that doesn't require any tool usage....

🤖 Answer: The capital of France is Paris.


## Test 4: Compare two cities — LLM may call the tool multiple times

In [ ]:
## Cell 6: Multi-tool — LLM should call weather for both cities
answer = host.chat("Compare the weather in New York and London right now. Which is warmer?")


👤 User: Compare the weather in New York and London right now. Which is warmer?

💭 Thinking: The user wants to compare the current weather in New York and London to see which is warmer. I need to get the current weather for both cities using the get_current_weather function. I'll call it twice, once for each city.

For the location parameter, I should use city names. I'll use "New York" and...

🔧 Tool call [1]: get_current_weather({"location": "New York"})
📦 Result: {"location": "New York, United States", "coordinates": {"lat": 40.71427, "lon": -74.00597}, "temperature": 49.5, "feels_like": 44.2, "unit": "fahrenheit", "condition": "Clear sky", "humidity": 72, "wind_speed": 7.3, "wind_direction": 153, "timezone": "America/New_York"}

🔧 Tool call [1]: get_current_weather({"location": "London"})
📦 Result: {"location": "London, United Kingdom", "coordinates": {"lat": 51.50853, "lon": -0.12574}, "temperature": 45.9, "feels_like": 40.6, "unit": "fahrenheit", "condition": "Mainly clear", "hum

## Cleanup
Close the MCP host and server connections.

In [4]:
## Cell 7: Cleanup — close MCP connections
host.stop()
print("✅ MCP host closed.")

⚠️ Shutdown warning: Attempted to exit cancel scope in a different task than it was entered in
✅ MCPHost stopped
✅ MCP host closed.
